In [1]:
import pandas as pd
import numpy as np
import evaluate
from tqdm import tqdm
tqdm.pandas()
from datasets import Dataset, DatasetDict
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer

# Load Data

In [2]:
df_cleantech = pd.read_json('/mnt/hdd01/PATSTAT Working Directory/PATSTAT/df_patstat_cleantech_granted_abstract_metadata.json')
df_cleantech['label'] = 1
df_non_cleantech = pd.read_json('/mnt/hdd01/PATSTAT Working Directory/PATSTAT/df_patstat_non_cleantech_granted_abstract_metadata.json')
df_non_cleantech['label'] = 0
df_cleantech = df_cleantech[df_cleantech['appln_abstract'] != '']
df_non_cleantech = df_non_cleantech[df_non_cleantech['appln_abstract'] != '']
df_cleantech.dropna(inplace=True)
df_non_cleantech.dropna(inplace=True)

In [3]:
# df = pd.concat([df_cleantech, df_non_cleantech], ignore_index=True)
df = pd.concat([df_cleantech.sample(10000, random_state=42), df_non_cleantech.sample(10000, random_state=42)], ignore_index=True)
df['appln_abstract'] = df['appln_abstract'].apply(lambda x: ' '.join(map(str, x)) if isinstance(x, list) else x)
df.rename(columns={'appln_abstract': 'text'}, inplace=True)
df = df[['text', 'label']]
df = df.sample(frac=1).reset_index(drop=True)

# Prepare Training

In [4]:
dataset = Dataset.from_pandas(df)

In [5]:
train_test_dataset = dataset.train_test_split(test_size=0.1)
train_val_dataset = train_test_dataset['train'].train_test_split(test_size=0.1)

In [6]:
final_datasets = DatasetDict({
    'train': train_val_dataset['train'],
    'validation': train_val_dataset['test'],
    'test': train_test_dataset['test']
})

## Instantiate Model

In [7]:
model_name = 'climatebert/distilroberta-base-climate-f'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

/home/thiesen/Documents/Projekt_EDV-TEK/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at climatebert/distilroberta-base-climate-f and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=512)

In [9]:
tokenized_datasets = final_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/16200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [10]:
training_args = TrainingArguments(
    output_dir='/home/thiesen/Documents/Projekt_EDV-TEK/',
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    # logging_dir='./logs',  # Logs will be saved here
    logging_steps=10,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [11]:
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [12]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

/home/thiesen/Documents/Projekt_EDV-TEK/venv/lib/python3.10/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False)
  warnings.warn(


In [13]:
# Start training
trainer.train()

You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.484000,0.439774,0.803333
2,0.405400,0.437886,0.800000
3,0.286800,0.602439,0.741111
4,0.168600,0.717226,0.788333
5,0.128600,0.879242,0.779444
6,0.108600,1.187563,0.774444
7,0.058300,1.376382,0.780000
8,0.017200,1.531093,0.786667
9,0.014800,1.617889,0.787778
10,0.035500,1.677229,0.786667


TrainOutput(global_step=5070, training_loss=0.15942314885847392, metrics={'train_runtime': 2805.4423, 'train_samples_per_second': 57.745, 'train_steps_per_second': 1.807, 'total_flos': 2.1459718582272e+16, 'train_loss': 0.15942314885847392, 'epoch': 10.0})